In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import spectrum as sp
import matplotlib.pyplot as plt

In [ ]:
import numpy as np

DROPLET_ACQS = range(48, 68)  # acq68 was still running, its log is not readable yet
AXES = ["a_mp1_x", "a_mp1_y", "a_mp1_z", "a_mp1_azim", "a_mp1_yaw"]

d = daq.manipulator_log(DROPLET_ACQS)  # one row per logged sample, local time
d.head()

In [ ]:
# every motor against time
fig, ax = plt.subplots(len(AXES), 1, figsize=(11, 11), sharex=True)
for a, name in zip(ax, AXES):
    s = d[["time", name]].dropna()
    a.plot(s.time, s[name], ".", ms=1)
    a.set_ylabel(name.removeprefix(daq.MANIPULATOR))
ax[-1].set_xlabel("local time")
fig.autofmt_xdate()
fig.tight_layout()

In [ ]:
# x against z at each y, y = 4 -> 50 mm ("down") and 50 -> 10 mm ("up"); one mean position per acquisition
# acq62 is the first run at y = 50 mm, acq63 the second, which starts the way up
DOWN = [51, 54, 56, 58, 60, 61, 62]
UP = [63, 64, 65, 67]
m = d.groupby("acq")[["a_mp1_x", "a_mp1_y", "a_mp1_z"]].mean()

# y labels sit perpendicular to the jet line, down-leg labels on one side, up-leg on the other
LABEL_OFFSET = {"down": ((-9, 7), "right"), "up": ((9, -9), "left")}

fig, ax = plt.subplots(figsize=(6, 5))
for acqs, color, label in [(DOWN, "C0", "down"), (UP, "C3", "up")]:
    g = m.loc[acqs]
    ax.plot(g.a_mp1_x, g.a_mp1_z, "o-", c=color, label=label)
    offset, ha = LABEL_OFFSET[label]
    for _, r in g.assign(y=g.a_mp1_y.round()).drop_duplicates("y").iterrows():  # one label per y
        ax.annotate(f"{r.y:.0f}", (r.a_mp1_x, r.a_mp1_z), xytext=offset, textcoords="offset points",
                    ha=ha, va="center", fontsize=8, color=color)
ax.set(xlabel="x (mm)", ylabel="z (mm)")
ax.margins(0.1)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# tilt of the jet (down leg from y = 10 mm) and up-minus-down offset at the same y
dn = m.loc[DOWN].iloc[1:]
for axis in ["a_mp1_x", "a_mp1_z"]:
    slope = np.polyfit(dn.a_mp1_y, dn[axis], 1)[0]
    print(f"d{axis[-1]}/dy = {slope:.4f} mm/mm  ({np.degrees(np.arctan(slope)):.1f} deg)")

for y in [20, 30, 40]:
    down = m[(m.a_mp1_y.round() == y) & m.index.isin(DOWN)].mean()
    up = m[(m.a_mp1_y.round() == y) & m.index.isin(UP)].mean()
    print(f"y={y}: up - down  dx = {1e3 * (up.a_mp1_x - down.a_mp1_x):+.0f} um, dz = {1e3 * (up.a_mp1_z - down.a_mp1_z):+.1f} um")

# drift of x within one acquisition
for acq in [56, 60, 62, 63, 67]:
    s = d[d.acq == acq].a_mp1_x.dropna()
    print(f"acq{acq}: x {1e3 * (s.iloc[-1] - s.iloc[0]):+.0f} um from first to last sample")

In [ ]:
def fit_jet(y, x, z, degree=1):
    """Fit the jet position x(y), z(y) with polynomials of the given degree.

    Returns predict(y) -> (x, z) in mm, for a scalar or an array of y (mm). Trust it
    only inside the fitted y range: a polynomial extrapolates badly.
    """
    px = np.polyfit(y, x, degree)
    pz = np.polyfit(y, z, degree)

    def predict(y_new):
        y_new = np.asarray(y_new, dtype=float)
        return np.polyval(px, y_new), np.polyval(pz, y_new)

    return predict


fit_pts = m.loc[DOWN + UP]  # both legs, one mean position per acquisition
for degree in [1, 2]:
    predict = fit_jet(fit_pts.a_mp1_y, fit_pts.a_mp1_x, fit_pts.a_mp1_z, degree)
    x_fit, z_fit = predict(fit_pts.a_mp1_y)
    rms_x = 1e3 * np.sqrt(np.mean((x_fit - fit_pts.a_mp1_x) ** 2))
    rms_z = 1e3 * np.sqrt(np.mean((z_fit - fit_pts.a_mp1_z) ** 2))
    print(f"degree {degree}: rms residual x {rms_x:.0f} um, z {rms_z:.0f} um")

predict = fit_jet(fit_pts.a_mp1_y, fit_pts.a_mp1_x, fit_pts.a_mp1_z, degree=1)
for y in [10, 20, 30, 40, 50]:
    x_p, z_p = predict(y)
    print(f"y = {y:2d} mm -> x = {x_p:+.3f} mm, z = {z_p:.3f} mm")